# PhoBERT MLM Continued Pretraining (Domain Adaptation)

This notebook implements the domain adaptation stage of the **News Sentiment Analysis** project. It Continued-Pretrains `vinai/phobert-base` on the **ViFiC financial corpus** using Masked Language Modeling (MLM).

### Recommended Environments:
- **Google Colab** (with T4 GPU or TPU)
- **Kaggle Notebooks** (with P100/T4x2 GPU or TPU v3-8)

In [ ]:
# Install dependencies
!pip install -q transformers tf-keras underthesea pandas numpy tensorflow

In [ ]:
import os
import re
import json
import numpy as np
import pandas as pd
import tensorflow as tf
import tf_keras as keras
from transformers import AutoTokenizer, TFRobertaForMaskedLM
from pathlib import Path
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

## 1. Data Preparation
We join the `title` and the first 300 characters of the `body` to define the input representation for each article:
$$\text{input\_text} = \text{\{title\}} + \text{" . "} + \text{\{body\_lead\}}$$
We then run `underthesea` for Vietnamese word segmentation and filter out entries that fall outside our length range of $[5, 220]$ tokens.

In [ ]:
def body_lead(body: str, max_chars: int = 300) -> str:
    if pd.isna(body):
        return ""
    body = str(body).strip()
    return body[:max_chars]

def build_input_text(title: str, body_lead: str) -> str:
    t = str(title).strip() if not pd.isna(title) else ""
    b = str(body_lead).strip() if not pd.isna(body_lead) else ""
    if t and b:
        return f"{t} . {b}"
    return t or b

from underthesea import word_tokenize
def segment_text(text: str) -> str:
    if not text:
        return ""
    text = re.sub(r"\s+", " ", text).strip()
    # Segment syllables with underscores
    return word_tokenize(text, format="text")

### Load Raw ViFiC CSV Dataset
Configure the path below to point to your raw ViFiC file (e.g. from Kaggle dataset or uploaded on Google Drive).

In [ ]:
# Configurable paths
VIFIC_RAW_CSV = "vific_raw.csv" # Change this path as needed
PRETRAINED_OUTPUT_CSV = "vific_pretraining.csv"

if os.path.exists(VIFIC_RAW_CSV):
    print(f"Loading raw ViFiC from {VIFIC_RAW_CSV}...")
    df = pd.read_csv(VIFIC_RAW_CSV)
    
    # Construct lead and segment
    print("Cleaning, segmenting and counting tokens...")
    df["body_lead"] = df["body"].map(lambda x: body_lead(x, max_chars=300))
    df["input_text"] = [build_input_text(t, b) for t, b in zip(df["title"], df["body_lead"])]
    df["input_text_segmented"] = df["input_text"].map(segment_text)
    df["token_count"] = df["input_text_segmented"].map(lambda x: len(str(x).split()))
    
    # Filter tokens between 5 and 220
    filtered_df = df[df["token_count"].between(5, 220)].copy().reset_index(drop=True)
    print(f"Total items: {len(df)} | Retained: {len(filtered_df)} (dropped {len(df)-len(filtered_df)})")
    
    filtered_df.to_csv(PRETRAINED_OUTPUT_CSV, index=False)
    print(f"Saved dataset to {PRETRAINED_OUTPUT_CSV}")
else:
    print(f"Raw dataset not found at {VIFIC_RAW_CSV}. Creating a mock dummy dataset for validation...")
    # Create a small dummy dataset so this notebook is executable right away
    mock_data = pd.DataFrame([
        {"title": "Thị trường chứng khoán tăng mạnh", "body": "Các mã cổ phiếu tài chính đồng loạt tăng kịch trần giúp chỉ số VN-Index vượt cản thành công."},
        {"title": "Báo cáo tài chính quý 4 nhiều khởi sắc", "body": "Nhiều doanh nghiệp công bố lợi nhuận ròng tăng vượt kỳ vọng đẩy thanh khoản lên cao."}
    ] * 50)
    mock_data["body_lead"] = mock_data["body"].map(lambda x: body_lead(x, max_chars=300))
    mock_data["input_text"] = [build_input_text(t, b) for t, b in zip(mock_data["title"], mock_data["body_lead"])]
    mock_data["input_text_segmented"] = mock_data["input_text"].map(segment_text)
    mock_data["token_count"] = mock_data["input_text_segmented"].map(lambda x: len(str(x).split()))
    filtered_df = mock_data[mock_data["token_count"].between(5, 220)].copy().reset_index(drop=True)
    filtered_df.to_csv(PRETRAINED_OUTPUT_CSV, index=False)
    print(f"Saved dummy dataset to {PRETRAINED_OUTPUT_CSV}")

## 2. MLM Input Masking
This logic randomly masks 15% of the input tokens, replacing them with either `<mask >` (80%), a random token (10%), or keeping them unchanged (10%). Non-masked tokens are labelled `-100` so they don't contribute to the loss function.

In [ ]:
def mask_tokens(inputs: np.ndarray, tokenizer: AutoTokenizer, mlm_probability: float = 0.15) -> tuple[np.ndarray, np.ndarray]:
    labels = np.copy(inputs)
    probability_matrix = np.full(labels.shape, mlm_probability)
    
    # Do not mask special tokens
    special_tokens_mask = []
    for row in labels.tolist():
        special_tokens_mask.append(tokenizer.get_special_tokens_mask(row, already_has_special_tokens=True))
    special_tokens_mask = np.array(special_tokens_mask, dtype=bool)
    probability_matrix[special_tokens_mask] = 0.0

    masked_indices = np.random.binomial(1, probability_matrix).astype(bool)
    labels[~masked_indices] = -100 # Ignore in loss

    # 80% replace with mask token
    indices_replaced = np.random.binomial(1, 0.8, size=labels.shape).astype(bool) & masked_indices
    inputs[indices_replaced] = tokenizer.mask_token_id

    # 10% replace with random token
    indices_random = np.random.binomial(1, 0.5, size=labels.shape).astype(bool) & masked_indices & ~indices_replaced
    random_words = np.random.randint(low=0, high=len(tokenizer), size=labels.shape)
    inputs[indices_random] = random_words[indices_random]
    
    # Remaining 10% stay unchanged
    return inputs, labels

## 3. Dataset Setup & Tokenization

In [ ]:
def build_dataset(features: dict[str, np.ndarray], labels: np.ndarray, batch_size: int, shuffle: bool) -> tf.data.Dataset:
    dataset = tf.data.Dataset.from_tensor_slices((features, labels))
    if shuffle:
        dataset = dataset.shuffle(min(len(labels), 4096))
    return dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

In [ ]:
df_prep = pd.read_csv(PRETRAINED_OUTPUT_CSV)
texts = df_prep["input_text_segmented"].dropna().astype(str).tolist()

# Load base tokenizer
base_model = "vinai/phobert-base"
tokenizer = AutoTokenizer.from_pretrained(base_model)

print(f"Tokenizing {len(texts)} entries...")
max_length = 256
encodings = tokenizer(
    texts,
    truncation=True,
    max_length=max_length,
    padding="max_length",
    return_tensors="np"
)

# Validation split (10%)
total_rows = len(encodings["input_ids"])
val_split = 0.1
val_size = max(1, int(total_rows * val_split))
train_size = max(1, total_rows - val_size)

input_ids = encodings["input_ids"]
attention_mask = encodings["attention_mask"]

train_inputs, val_inputs = input_ids[:train_size], input_ids[train_size:]
train_attention, val_attention = attention_mask[:train_size], attention_mask[train_size:]

# Mask tokens for train and val
train_masked, train_labels = mask_tokens(train_inputs.copy(), tokenizer)
val_masked, val_labels = mask_tokens(val_inputs.copy(), tokenizer)

train_features = {
    "input_ids": train_masked,
    "attention_mask": train_attention,
    "labels": train_labels
}
val_features = {
    "input_ids": val_masked,
    "attention_mask": val_attention,
    "labels": val_labels
}

## 4. Accelerator Detection (GPU/TPU Strategy)
Detects if TPU or GPU is available to scale the training strategy accordingly.

In [ ]:
try:
    resolver = tf.distribute.cluster_resolver.TPUClusterResolver()
    tf.config.experimental_connect_to_cluster(resolver)
    tf.tpu.experimental.initialize_tpu_system(resolver)
    strategy = tf.distribute.TPUStrategy(resolver)
    print(f"Using TPU distribution strategy: {resolver.master()}")
except ValueError:
    gpus = tf.config.list_physical_devices("GPU")
    if gpus:
        strategy = tf.distribute.OneDeviceStrategy(device="/gpu:0")
        print(f"Using GPU distribution strategy: {gpus}")
    else:
        strategy = tf.distribute.get_strategy()
        print("No accelerators found, using default CPU strategy")

## 5. Continued Pretraining Model Training
Initializes `TFRobertaForMaskedLM` from PyTorch weights and fits the model.

In [ ]:
# Configs
epochs = 3
batch_size = 16 # Increase to 32 or 64 if training on high-VRAM GPU/TPU
learning_rate = 2e-5
output_dir = "./phobert-financial-mlm-adapted"

train_dataset = build_dataset(train_features, train_labels, batch_size, shuffle=True)
val_dataset = build_dataset(val_features, val_labels, batch_size, shuffle=False)

with strategy.scope():
    model = TFRobertaForMaskedLM.from_pretrained(base_model, from_pt=True)
    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer)

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)
]

print("Starting MLM continued pretraining...")
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=epochs,
    callbacks=callbacks,
    verbose=1
)

# Save adaptation outputs
os.makedirs(output_dir, exist_ok=True)
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Domain-adapted model and tokenizer successfully saved to: {output_dir}")